# Econ 148 — Machine Learning for Economists
## Part 1: Bias-Variance Tradeoff 
### Data: Current Population Survey (CPS), 1978 & 1985

**The central question:** Can we predict wages better than the classic Mincer equation?  
**The deeper question:** What does "better" even mean — and what can go wrong?

---

**Variables in the CPS data:**
| Variable | Description |
|---|---|
| `lwage` | Log hourly wage |
| `educ` | Years of education |
| `exper` | Years of work experience |
| `female` | 1 if female |
| `married` | 1 if married |
| `union` | 1 if union member |
| `nonwhite` | 1 if nonwhite |
| `south` | 1 if lives in south |
| `y85` | 1 if observation is from 1985 |

In [ ]:
try: import wooldridge
except ImportError:
    !pip install wooldridge -q
    import wooldridge

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, RidgeCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Load CPS data
df = wooldridge.data('cps78_85')
print(f"Dataset shape: {df.shape}")
print(f"\nSummary statistics:")
df[['lwage','educ','exper','female','married','union','nonwhite','south']].describe().round(2)

---
## Part 1: The Bias-Variance Tradeoff

### 1.1 The Classic Mincer Equation (What You Already Know)

The Mincer earnings equation is:

$$\log(wage_i) = \beta_0 + \beta_1 \cdot educ_i + \beta_2 \cdot exper_i + \beta_3 \cdot exper_i^2 + \varepsilon_i$$

This is a great model for **inference** — we trust $\hat{\beta}_1$ as an estimate of returns to education.  
But is it the best model for **prediction**?

In [ ]:
import statsmodels.formula.api as smf

# Classic Mincer OLS
mincer = smf.ols('lwage ~ educ + exper + expersq', data=df).fit()
print(mincer.summary().tables[1])

### 1.2 The ML Mindset: Train vs. Test Error

In econometrics, we evaluate models on the **same data** we used to fit them (in-sample R²).  
In ML, we hold out a **test set** the model never sees — this is the true measure of prediction quality.

> **Key insight:** A model can fit the training data perfectly and still predict terribly on new data. This is **overfitting**.

In [ ]:
# Set up features and outcome
X_base = df[['educ', 'exper']].values
y = df['lwage'].values

# Split: 70% train, 30% test — the model never sees the test set
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.3, random_state=42
)

print(f"Training observations:  {len(X_train)}")
print(f"Test observations:      {len(X_test)}")
print(f"\nThe model is trained on {len(X_train)} observations.")
print(f"Its prediction quality is evaluated on the {len(X_test)} held-out observations.")

### 1.3 Polynomial Mincer: Adding Flexibility

What if the relationship between experience and wages isn't quadratic — it's more complex?  
We can fit higher-degree polynomials of `experience`. Let's see what happens as we increase the degree.

In [ ]:
# Fit polynomials of increasing degree using experience only
# (isolating one variable makes the visual cleaner)
exper_train = X_train[:, 1].reshape(-1, 1)
exper_test  = X_test[:, 1].reshape(-1, 1)

degrees = [1, 2, 4, 8, 12, 16]
train_r2, test_r2, train_mse, test_mse = [], [], [], []

for deg in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('ols',  LinearRegression())
    ])
    pipe.fit(exper_train, y_train)
    
    train_r2.append(r2_score(y_train, pipe.predict(exper_train)))
    test_r2.append(r2_score(y_test,  pipe.predict(exper_test)))
    train_mse.append(mean_squared_error(y_train, pipe.predict(exper_train)))
    test_mse.append(mean_squared_error(y_test,  pipe.predict(exper_test)))

results = pd.DataFrame({
    'Degree': degrees,
    'Train R²': [f"{r:.4f}" for r in train_r2],
    'Test R²':  [f"{r:.4f}" for r in test_r2],
    'Train MSE': [f"{m:.4f}" for m in train_mse],
    'Test MSE':  [f"{m:.4f}" for m in test_mse],
})
print(results.to_string(index=False))

In [ ]:
# ── Figure 1: The Bias-Variance Tradeoff ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: R² curves
ax = axes[0]
ax.plot(degrees, train_r2, 'o-', color='steelblue',  lw=2.5, ms=8, label='Training R²')
ax.plot(degrees, test_r2,  's--', color='tomato',    lw=2.5, ms=8, label='Test R²')
ax.axvline(x=2, color='gray', linestyle=':', lw=1.5, label='Classic Mincer (degree 2)')
ax.set_xlabel('Polynomial Degree of Experience')
ax.set_ylabel('R²')
ax.set_title('Train vs. Test R²\n(higher degree = more flexible model)', fontsize=12)
ax.legend()
ax.annotate('Overfitting\nregion', xy=(10, test_r2[-2]),
            xytext=(11, test_r2[-2]+0.03),
            arrowprops=dict(arrowstyle='->', color='tomato'),
            color='tomato', fontsize=10)

# Right: Fitted curves on raw data (train + test)
ax = axes[1]
exper_all = np.linspace(df['exper'].min(), df['exper'].max(), 300).reshape(-1, 1)

colors = ['#2ecc71', 'steelblue', 'darkorange', 'tomato']
highlight_degrees = [1, 2, 4, 16]

ax.scatter(exper_test, y_test, alpha=0.15, color='gray', s=15, label='Test data')

for deg, col in zip(highlight_degrees, colors):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('ols',  LinearRegression())
    ])
    pipe.fit(exper_train, y_train)
    preds = pipe.predict(exper_all)
    label = f'Degree {deg}' + (' (Mincer)' if deg == 2 else '')
    ax.plot(exper_all, preds, color=col, lw=2.2, label=label)

ax.set_xlabel('Years of Experience')
ax.set_ylabel('Log Wage (predicted)')
ax.set_title('Fitted Wage-Experience Profiles\n(degree 16 goes wild!)', fontsize=12)
ax.set_ylim(df['lwage'].min() - 0.3, df['lwage'].max() + 0.5)
ax.legend(fontsize=9)

plt.suptitle('Figure 1: The Bias-Variance Tradeoff — CPS Wage Data', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_bias_variance.png', bbox_inches='tight', dpi=130)
plt.show()

print("\n💡 Notice: Training R² keeps rising as degree increases.")
print("   Test R² peaks around degree 2–4 and then FALLS.")
print("   Degree 16 memorizes the training data — it's useless for prediction.")

### 1.4 Understanding the Tradeoff

| | Low Flexibility (degree 1) | Just Right (degree 2–4) | Too Flexible (degree 16) |
|---|---|---|---|
| **Bias** | High — misses real patterns | Low | Very low |
| **Variance** | Low — stable across samples | Medium | Very high — wiggles everywhere |
| **Training R²** | Low | Medium | Near 1.0 |
| **Test R²** | Low | **Highest** | Low or negative |
| **Econometric name** | Underfitting | Good fit | Overfitting |

> **The fundamental insight:** Expected prediction error = Bias² + Variance + Irreducible noise.  
> Reducing bias by adding complexity *increases* variance. There's no free lunch.

### 1.5 Cross-Validation: A Smarter Way to Pick Model Complexity

Instead of a single train/test split, **k-fold cross-validation** averages over $k$ different splits — giving a much more stable estimate of out-of-sample performance.

In [ ]:
# 5-fold cross-validation across polynomial degrees
degrees_cv = list(range(1, 18))
cv_scores_mean, cv_scores_std = [], []

for deg in degrees_cv:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('ols',  LinearRegression())
    ])
    scores = cross_val_score(pipe, exper_train, y_train,
                             cv=5, scoring='r2')
    cv_scores_mean.append(scores.mean())
    cv_scores_std.append(scores.std())

cv_mean = np.array(cv_scores_mean)
cv_std  = np.array(cv_scores_std)
best_degree = degrees_cv[np.argmax(cv_mean)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(degrees_cv, cv_mean, 'o-', color='steelblue', lw=2.5, ms=7, label='CV mean R²')
ax.fill_between(degrees_cv, cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.2, color='steelblue', label='±1 SD across folds')
ax.axvline(x=best_degree, color='tomato', linestyle='--', lw=2,
           label=f'Best degree = {best_degree} (CV-selected)')
ax.set_xlabel('Polynomial Degree of Experience')
ax.set_ylabel('5-Fold Cross-Validated R²')
ax.set_title('Figure 2: Cross-Validation Selects the Optimal Complexity\n(CPS Wage Data)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('fig2_crossval.png', bbox_inches='tight', dpi=130)
plt.show()
print(f"\n✅ Cross-validation selects degree = {best_degree}")
print("   This is close to the classic Mincer quadratic — CV rediscovers good econometric practice!")